# Data Mining Assignment: Spark Data Lake (Bronze to Silver)

**Name:** Aman Sharma | **Roll No:** MSD25024
**Dataset:** `sales.csv` | **Environment:** Google Colab + PySpark

This notebook builds a two-layer data lake:

- **Bronze**: the raw `sales.csv`, copied once and never changed.
- **Silver**: the same data after basic cleaning with Spark.

Steps: set up folders, load and inspect the raw data, clean it, save it to Silver, then read it back to check.

## 1. Setup
Install PySpark and create the folder layout. The raw file is downloaded from the class repository if it is not already in Bronze.

In [1]:
!pip install -q pyspark

In [2]:
import os, glob, shutil, urllib.request

BRONZE = "data_lake/bronze/sales.csv"
SILVER_DIR = "data_lake/silver/preprocessed_sales"
SILVER_CSV = "data_lake/silver/sales_silver.csv"
SOURCE_URL = ("https://raw.githubusercontent.com/necromancersanju-netizen/"
              "data-mining-data-lake-assignment/main/dataset/sales.csv")

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)
if not os.path.exists(BRONZE):
    urllib.request.urlretrieve(SOURCE_URL, BRONZE)

for root, dirs, files in os.walk("data_lake"):
    print(root, files)

data_lake []
data_lake/silver []
data_lake/silver/preprocessed_sales []
data_lake/bronze ['sales.csv']


## 2. Load and inspect the Bronze data
ANSI mode is turned off so that a bad date becomes `null` instead of stopping the job.

In [3]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("sales-data-lake")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .getOrCreate())

raw = spark.read.csv(BRONZE, header=True, inferSchema=True)
raw.printSchema()
raw.show(5, truncate=False)
print("Total records:", raw.count())

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

### 2.1 Find the problems
Every count below comes from a Spark query on the raw data.

In [4]:
print("Exact duplicate rows:", raw.count() - raw.dropDuplicates().count())

print("Null values per column:")
raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw.columns]).show()

Exact duplicate rows: 10
Null values per column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [5]:
# Text columns whose values have extra spaces or mixed capitalisation
TITLE_COLS = ["customer_name", "category", "city", "state"]
raw.select([F.sum((F.col(c) != F.initcap(F.trim(c))).cast("int")).alias(c) for c in TITLE_COLS]).show()

raw.groupBy("category").count().orderBy("category").show()
raw.groupBy("city").count().orderBy("city").show(30)

+-------------+--------+----+-----+
|customer_name|category|city|state|
+-------------+--------+----+-----+
|            3|       6|   9|    0|
+-------------+--------+----+-----+

+---------------+-----+
|       category|count|
+---------------+-----+
|         Beauty|  136|
|          Books|  128|
|       CLOTHING|    1|
|       Clothing|  108|
|    ELECTRONICS|    1|
|    Electronics|  103|
|      Furniture|  132|
|        Grocery|  136|
|Home Appliances|  115|
|         SPORTS|    1|
|         Sports|  136|
|         beauty|    2|
|         sports|    1|
+---------------+-----+

+-----------+-----+
|       city|count|
+-----------+-----+
|       NULL|    8|
| Hyderabad |    2|
|    Mumbai |    1|
|  Bengaluru|   91|
|    Chennai|   88|
| Coimbatore|   81|
|      DELHI|    2|
|      Delhi|   81|
|  HYDERABAD|    1|
|  Hyderabad|   94|
|     JAIPUR|    1|
|     Jaipur|   84|
|    Kolkata|   81|
|    Lucknow|   90|
|     Mumbai|  106|
|       Pune|   82|
| Vijayawada|  105|
| coimbato

In [6]:
# Numbers that cannot be right: no item, a free or negative price, or a discount outside 0-100
raw.select(
    F.sum((F.col("quantity") <= 0).cast("int")).alias("bad_quantity"),
    F.sum((F.col("unit_price") <= 0).cast("int")).alias("bad_unit_price"),
    F.sum((~F.col("discount_percent").between(0, 100)).cast("int")).alias("bad_discount"),
).show()

+------------+--------------+------------+
|bad_quantity|bad_unit_price|bad_discount|
+------------+--------------+------------+
|          12|             8|           8|
+------------+--------------+------------+



In [7]:
# Dates: the expected format is yyyy-MM-dd. Try the other formats seen in the file; what still fails is invalid.
DATE_FORMATS = ["yyyy-MM-dd", "yyyy/MM/dd", "MM-dd-yyyy", "dd/MM/yyyy"]
parsed = F.coalesce(*[F.to_date("order_date", f) for f in DATE_FORMATS])

dates = raw.select("order_date", parsed.alias("parsed"))
dates.filter(~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$")).show()
print("Unparseable dates:", dates.filter(F.col("parsed").isNull()).count())

+----------+----------+
|order_date|    parsed|
+----------+----------+
|31/02/2026|      NULL|
|2025/07/14|2025-07-14|
|not-a-date|      NULL|
|09-15-2026|2026-09-15|
+----------+----------+

Unparseable dates: 4


## 3. Clean the data (Silver)

1. Drop exact duplicate rows. Rows that share an `order_id` but differ elsewhere are kept, since they may be real orders.
2. Trim spaces in every text column and title-case names, categories, cities and states.
3. Fill missing `customer_name`, `city` and `payment_method` with `Unknown` so the rows are kept.
4. Set impossible `quantity`, `unit_price` and `discount_percent` values to `null` instead of dropping the row.
5. Convert dates to `yyyy-MM-dd`; dates that cannot be parsed (e.g. `31/02/2026`, `not-a-date`) become `null`.

In [8]:
silver = raw.dropDuplicates()

for c, t in silver.dtypes:
    if t == "string":
        silver = silver.withColumn(c, F.trim(c))
for c in TITLE_COLS:
    silver = silver.withColumn(c, F.initcap(c))

silver = (silver
    .fillna("Unknown", subset=["customer_name", "city", "payment_method"])
    .withColumn("quantity", F.when(F.col("quantity") > 0, F.col("quantity")))
    .withColumn("unit_price", F.when(F.col("unit_price") > 0, F.col("unit_price")))
    .withColumn("discount_percent", F.when(F.col("discount_percent").between(0, 100), F.col("discount_percent")))
    .withColumn("order_date", parsed)
    .orderBy("order_id"))

silver.show(5, truncate=False)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080      |12              |UPI           |Pune      |Maharashtra   |2025-05-06|Shipped     |
|100004  |C0051      |

### 3.1 Save to Silver
Spark writes a folder of part files. One partition is used so the folder holds a single CSV, which is also copied to `sales_silver.csv` for submission.

In [9]:
silver.coalesce(1).write.mode("overwrite").option("header", True).csv(SILVER_DIR)
shutil.copyfile(glob.glob(f"{SILVER_DIR}/part-*.csv")[0], SILVER_CSV)
print(os.listdir(SILVER_DIR))

['_SUCCESS', '.part-00000-e3201cd7-47f0-4e3d-9396-1da4ef80feb0-c000.csv.crc', 'part-00000-e3201cd7-47f0-4e3d-9396-1da4ef80feb0-c000.csv', '._SUCCESS.crc']


## 4. Verify

In [10]:
check = spark.read.csv(SILVER_DIR, header=True, inferSchema=True)
print("Rows before:", raw.count(), "| Rows after:", check.count())
check.printSchema()
check.show(5, truncate=False)

print("Nulls left per column:")
check.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in check.columns]).show()

Rows before: 1000 | Rows after: 990
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+--------

## Summary

- **Bronze** `data_lake/bronze/sales.csv` is the untouched raw file.
- **Silver** `data_lake/silver/preprocessed_sales/` (plus `sales_silver.csv`) holds the cleaned data.
- Done: removed exact duplicates, trimmed and title-cased text, filled missing text with `Unknown`,
  nulled impossible quantities, prices and discounts, and standardised dates to `yyyy-MM-dd` (invalid ones set to `null`).
- Kept on purpose: rows with a repeated `order_id` but different details, and high but positive prices.